In [71]:
#Task 4.1

import sqlite3
conn = sqlite3.connect("school.db") #Connect
cursor = conn.cursor() #Use cursor object to execute commands
cursor.execute("DROP TABLE IF EXISTS 'People' ") #If exist, drop so dont overwrite
cursor.execute("""CREATE TABLE "People" (
	"PersonID"	INTEGER,
	"FullName"	TEXT,
	"DateOfBirth"	TEXT,
	"ScreenName"	TEXT,
	"IsAdult"	INTEGER,
	PRIMARY KEY("PersonID" AUTOINCREMENT)
);""")
conn.commit()
conn.close()

In [72]:
#Task 4.2 

class Person:
    def __init__(self, full_name, dob):
        self.full_name = full_name
        self.date_of_birth = dob

    def is_adult(self):
        year = int(self.date_of_birth[:4])
        age = 2020 - year
        if age > 18:
            return True
        return False

    def screen_name(self):
        name = ''
        for i in self.full_name:
            if i.upper() in 'ABCDEFGHIJKLMNOPQRSTUVWXYZ':
                name += i
        screen_name = name + self.date_of_birth[5:7] + self.date_of_birth[8:10]
        return screen_name

#Test Cases
person = Person("Adam","2001-06-01")
print(person.is_adult())
print(person.screen_name())

True
Adam0601


In [73]:
#Task 4.2 PART 2

class Staff(Person):
    def screen_name(self): #Auto defines parameters, since inherited automtaically 
        name = ''
        for i in self.full_name:
            if i.upper() in 'ABCDEFGHIJKLMNOPQRSTUVWXYZ':
                name += i
        screen_name = name + self.date_of_birth[5:7] + self.date_of_birth[8:10] + "Staff"
        return screen_name
    def is_adult(self):
        return True

class Student(Person):
    def screen_name(self): #Auto defines parameters, since inherited automtaically 
        name = ''
        for i in self.full_name:
            if i.upper() in 'ABCDEFGHIJKLMNOPQRSTUVWXYZ':
                name += i
        screen_name = name + self.date_of_birth[5:7] + self.date_of_birth[8:10] + "Student"
        return screen_name
        
    def is_adult(self):
        return False

In [74]:
#Task 4.2 PART 3

with open("people.txt", "r") as file: #file closes automatically
    lines = file.readlines()
    classes = [] #save all the classes
    for i in lines:
        part = i.strip().split(",")
        if part[2] == "Person":
            indiv = Person(part[0],part[1])
            classes.append(indiv)
            
        elif part[2] == "Staff":
            indiv = Staff(part[0],part[1])
            classes.append(indiv)
        else:
            indiv = Student(part[0],part[1])
            classes.append(indiv)
            
#checking if classes are saved
print(classes)

[<__main__.Person object at 0x000001AB9CADBB10>, <__main__.Person object at 0x000001AB9D2A2490>, <__main__.Staff object at 0x000001AB9D63BE00>, <__main__.Student object at 0x000001AB9CCC67B0>, <__main__.Staff object at 0x000001AB9D2A1590>, <__main__.Student object at 0x000001AB9D2A0E10>, <__main__.Student object at 0x000001AB9D2A1310>]


In [75]:
#Task 4.2 Last PART

import sqlite3
conn = sqlite3.connect("school.db")
cursor = conn.cursor()

count = 0
for c in classes:
    count += 1
    cursor.execute("""INSERT INTO People (PersonID,FullName,DateOfBirth,ScreenName,IsAdult)
                    VALUES(?,?,?,?,?)""", 
                   (count, c.full_name, c.date_of_birth,c.screen_name(),c.is_adult())) #Columns then values
conn.commit()
conn.close()

In [ ]:
#Task 4.3

from flask import Flask, render_template
import sqlite3



#exporting to web app
app = Flask(__name__)


@app.route('/')
def home():
    ptype = []
    #fetching the info from SQL database
    conn = sqlite3.connect("school.db")
    cursor = conn.cursor()
    cursor.execute("""SELECT * FROM People""")
    persondata = cursor.fetchall()
    conn.close()
    for data in persondata:
        if "Staff" in data[3]:
            ptype.append("Staff")
        elif "Student" in data[3]:
            ptype.append("Student")
        else:
            ptype.append("Person")
        
    return render_template("index.html",
                           data=persondata,
                           ptype=ptype)

if __name__ == "__main__":
    app.run(debug=True, use_reloader=False, port=5000)

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [16/Aug/2026 17:16:40] "GET / HTTP/1.1" 500 -
Traceback (most recent call last):
  File "C:\Users\thedr\miniconda3\Lib\site-packages\flask\app.py", line 1536, in __call__
    return self.wsgi_app(environ, start_response)
           ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\thedr\miniconda3\Lib\site-packages\flask\app.py", line 1514, in wsgi_app
    response = self.handle_exception(e)
  File "C:\Users\thedr\miniconda3\Lib\site-packages\flask\app.py", line 1511, in wsgi_app
    response = self.full_dispatch_request()
  File "C:\Users\thedr\miniconda3\Lib\site-packages\flask\app.py", line 919, in full_dispatch_request
    rv = self.handle_user_exception(e)
  File "C:\Users\thedr\miniconda3\Lib\site-packages\flask\app.py", line 917, in full_dispatch_request
    rv = self.dispatch_request()
  File "C:\Users\thedr\miniconda3\Lib\site-packages\flask\app.py", line 902, in dispatch_request
    return self.ensur